# 📊 Global Literacy & Education Trends: An Analytical Study
### Complete Data Collection + Cleaning + EDA + SQL Notebook
**Steps covered:**
1. Install & Import Libraries
2. Collect all 5 datasets from Our World in Data
3. Data Understanding & Cleaning
4. Feature Engineering
5. EDA (Univariate + Bivariate)
6. SQL Database Creation & Insertion
7. SQL Queries (13 total)
8. Download Database

---
## ⚙️ Step 1: Install & Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly --quiet
print('✅ Libraries installed!')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sqlite3
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('✅ All libraries imported!')

---
## 📥 Step 2: Data Collection (Our World in Data)

In [ ]:
# ── 2A. Adult Literacy Rate ──
url_adult = 'https://ourworldindata.org/grapher/literacy-rate-adults.csv?v=1&csvType=full&useColumnShortNames=true'
df_adult = pd.read_csv(url_adult)
print('Adult Literacy shape:', df_adult.shape)
print(df_adult.columns.tolist())
df_adult.head(3)

In [ ]:
# ── 2B. Youth Literacy Rate (Male & Female) ──
url_youth = 'https://ourworldindata.org/grapher/literacy-rate-of-young-men-and-women.csv?v=1&csvType=full&useColumnShortNames=true'
df_youth = pd.read_csv(url_youth)
print('Youth Literacy shape:', df_youth.shape)
print(df_youth.columns.tolist())
df_youth.head(3)

In [ ]:
# ── 2C. Illiterate Population ──
url_illit = 'https://ourworldindata.org/grapher/literate-and-illiterate-world-population.csv?v=1&csvType=full&useColumnShortNames=true'
df_illit = pd.read_csv(url_illit)
print('Illiteracy Population shape:', df_illit.shape)
print(df_illit.columns.tolist())
df_illit.head(3)

In [ ]:
# ── 2D. GDP per Capita ──
url_gdp = 'https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true'
df_gdp = pd.read_csv(url_gdp)
print('GDP shape:', df_gdp.shape)
print(df_gdp.columns.tolist())
df_gdp.head(3)

In [ ]:
# ── 2E. Average Years of Schooling ──
url_school = 'https://ourworldindata.org/grapher/literacy-rates-vs-average-years-of-schooling.csv?v=1&csvType=full&useColumnShortNames=true'
df_school = pd.read_csv(url_school)
print('Schooling shape:', df_school.shape)
print(df_school.columns.tolist())
df_school.head(3)

---
## 🧹 Step 3: Data Cleaning & Merging

In [ ]:
# ── Standardize column names across all dataframes ──
def standardize(df):
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    # Rename common OWID columns
    rename_map = {}
    for col in df.columns:
        if 'entity' in col:  rename_map[col] = 'country'
        if 'code'   in col:  rename_map[col] = 'code'
        if col == 'year':    rename_map[col] = 'year'
    df.rename(columns=rename_map, inplace=True)
    return df

df_adult  = standardize(df_adult)
df_youth  = standardize(df_youth)
df_illit  = standardize(df_illit)
df_gdp    = standardize(df_gdp)
df_school = standardize(df_school)

print('✅ Columns standardized')
for name, df in [('adult',df_adult),('youth',df_youth),('illit',df_illit),('gdp',df_gdp),('school',df_school)]:
    print(f'  {name}: {df.columns.tolist()}')

In [ ]:
# ── Filter years 1990-2023 ──
def filter_years(df, start=1990, end=2023):
    if 'year' in df.columns:
        df = df[(df['year'] >= start) & (df['year'] <= end)]
    return df.reset_index(drop=True)

df_adult  = filter_years(df_adult)
df_youth  = filter_years(df_youth)
df_illit  = filter_years(df_illit)
df_gdp    = filter_years(df_gdp)
df_school = filter_years(df_school)
print('✅ Year filter applied (1990-2023)')

In [ ]:
# ── Remove rows without country code (aggregates like 'World', 'Africa', etc.) ──
def remove_aggregates(df):
    if 'code' in df.columns:
        df = df[df['code'].notna() & (df['code'] != '')]
    return df.reset_index(drop=True)

df_adult  = remove_aggregates(df_adult)
df_youth  = remove_aggregates(df_youth)
df_illit  = remove_aggregates(df_illit)
df_gdp    = remove_aggregates(df_gdp)
df_school = remove_aggregates(df_school)
print('✅ Aggregates removed')

In [ ]:
# ── Build df_literacy: merge adult + youth ──
# Rename value columns clearly first
adult_val_col  = [c for c in df_adult.columns  if c not in ['country','code','year']]
youth_val_cols = [c for c in df_youth.columns  if c not in ['country','code','year']]
print('Adult value cols:', adult_val_col)
print('Youth value cols:', youth_val_cols)

df_adult_clean = df_adult[['country','code','year'] + adult_val_col].copy()
df_adult_clean.rename(columns={adult_val_col[0]: 'adult_literacy_rate'}, inplace=True)

df_youth_clean = df_youth[['country','code','year'] + youth_val_cols].copy()
# Rename youth columns
col_map = {}
for c in youth_val_cols:
    if 'male' in c.lower() and 'female' not in c.lower():
        col_map[c] = 'youth_literacy_male'
    elif 'female' in c.lower() or 'women' in c.lower():
        col_map[c] = 'youth_literacy_female'
    else:
        col_map[c] = 'youth_literacy_' + c
df_youth_clean.rename(columns=col_map, inplace=True)

df_literacy = pd.merge(df_adult_clean, df_youth_clean, on=['country','code','year'], how='outer')
df_literacy.dropna(subset=['adult_literacy_rate'], inplace=True)
df_literacy.drop_duplicates(subset=['country','year'], inplace=True)
df_literacy.reset_index(drop=True, inplace=True)
print(f'\n✅ df_literacy shape: {df_literacy.shape}')
df_literacy.head()

In [ ]:
# ── Build df_illiteracy ──
illit_val_cols = [c for c in df_illit.columns if c not in ['country','code','year']]
print('Illiteracy value cols:', illit_val_cols)

df_illiteracy = df_illit[['country','code','year'] + illit_val_cols].copy()
# Rename for clarity
rename_illit = {}
for c in illit_val_cols:
    cl = c.lower()
    if 'illiterate' in cl and 'male' in cl and 'female' not in cl:
        rename_illit[c] = 'illiterate_male'
    elif 'illiterate' in cl and 'female' in cl:
        rename_illit[c] = 'illiterate_female'
    elif 'illiterate' in cl:
        rename_illit[c] = 'illiterate_total'
    elif 'literate' in cl and 'il' not in cl:
        rename_illit[c] = 'literate_total'
df_illiteracy.rename(columns=rename_illit, inplace=True)

# Compute illiteracy % if both columns exist
if 'illiterate_total' in df_illiteracy.columns and 'literate_total' in df_illiteracy.columns:
    df_illiteracy['total_population'] = df_illiteracy['illiterate_total'] + df_illiteracy['literate_total']
    df_illiteracy['illiteracy_pct'] = (df_illiteracy['illiterate_total'] / df_illiteracy['total_population'] * 100).round(2)

df_illiteracy.drop_duplicates(subset=['country','year'], inplace=True)
df_illiteracy.reset_index(drop=True, inplace=True)
print(f'\n✅ df_illiteracy shape: {df_illiteracy.shape}')
df_illiteracy.head()

In [ ]:
# ── Build df_gdp_schooling ──
gdp_val_cols    = [c for c in df_gdp.columns    if c not in ['country','code','year']]
school_val_cols = [c for c in df_school.columns if c not in ['country','code','year']]
print('GDP cols:', gdp_val_cols)
print('Schooling cols:', school_val_cols)

df_gdp_clean    = df_gdp[['country','code','year'] + gdp_val_cols].copy()
df_school_clean = df_school[['country','code','year'] + school_val_cols].copy()

# Rename GDP column
df_gdp_clean.rename(columns={gdp_val_cols[0]: 'gdp_per_capita'}, inplace=True)

# Rename schooling column — pick the years of schooling column
for c in school_val_cols:
    if 'school' in c.lower() or 'year' in c.lower():
        df_school_clean.rename(columns={c: 'avg_years_schooling'}, inplace=True)
        break

df_gdp_schooling = pd.merge(df_gdp_clean, df_school_clean[['country','year','avg_years_schooling']],
                             on=['country','year'], how='outer')
df_gdp_schooling.drop_duplicates(subset=['country','year'], inplace=True)
df_gdp_schooling.dropna(subset=['gdp_per_capita'], inplace=True)
df_gdp_schooling.reset_index(drop=True, inplace=True)
print(f'\n✅ df_gdp_schooling shape: {df_gdp_schooling.shape}')
df_gdp_schooling.head()

In [ ]:
# ── Summary ──
print('='*50)
print('📊 Final DataFrame Summary')
print('='*50)
for name, df in [('df_literacy', df_literacy), ('df_illiteracy', df_illiteracy), ('df_gdp_schooling', df_gdp_schooling)]:
    print(f'\n{name}:')
    print(f'  Shape : {df.shape}')
    print(f'  Cols  : {df.columns.tolist()}')
    print(f'  Nulls : {df.isnull().sum().sum()}')

---
## ⚙️ Step 4: Feature Engineering

In [ ]:
# ── Feature 1: Literacy Gender Gap ──
if 'youth_literacy_male' in df_literacy.columns and 'youth_literacy_female' in df_literacy.columns:
    df_literacy['literacy_gender_gap'] = (df_literacy['youth_literacy_male'] - df_literacy['youth_literacy_female']).round(2)

# ── Feature 2: Youth Literacy Average ──
if 'youth_literacy_male' in df_literacy.columns and 'youth_literacy_female' in df_literacy.columns:
    df_literacy['youth_literacy_avg'] = ((df_literacy['youth_literacy_male'] + df_literacy['youth_literacy_female']) / 2).round(2)

# ── Feature 3: Literacy Growth Rate (year-over-year) ──
df_literacy = df_literacy.sort_values(['country','year'])
df_literacy['literacy_growth_rate'] = df_literacy.groupby('country')['adult_literacy_rate'].pct_change().mul(100).round(2)

# ── Feature 4: GDP per Schooling Year ──
if 'avg_years_schooling' in df_gdp_schooling.columns:
    df_gdp_schooling['gdp_per_schooling_year'] = (df_gdp_schooling['gdp_per_capita'] / df_gdp_schooling['avg_years_schooling'].replace(0, np.nan)).round(2)

# ── Feature 5: Education Index = (literacy/100 * schooling/15) ──
# Will be created on merged data later

print('✅ Features created!')
print('df_literacy columns:', df_literacy.columns.tolist())
print('df_gdp_schooling columns:', df_gdp_schooling.columns.tolist())

---
## 📊 Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# ── 5A. Univariate: Distribution of Adult Literacy Rate ──
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
df_literacy['adult_literacy_rate'].dropna().hist(bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Adult Literacy Rate')
plt.xlabel('Literacy Rate (%)')
plt.ylabel('Count')

plt.subplot(1,2,2)
df_literacy['adult_literacy_rate'].dropna().plot(kind='box', color='steelblue')
plt.title('Boxplot: Adult Literacy Rate')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5B. Univariate: GDP per Capita Distribution ──
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
df_gdp_schooling['gdp_per_capita'].dropna().hist(bins=40, color='darkorange', edgecolor='white')
plt.title('Distribution of GDP per Capita')
plt.xlabel('GDP per Capita (USD)')

plt.subplot(1,2,2)
np.log1p(df_gdp_schooling['gdp_per_capita'].dropna()).hist(bins=40, color='darkorange', edgecolor='white')
plt.title('Log Distribution of GDP per Capita')
plt.xlabel('Log(GDP per Capita)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5C. Bivariate: GDP vs Adult Literacy Rate ──
merged = pd.merge(df_literacy[['country','year','adult_literacy_rate']],
                  df_gdp_schooling[['country','year','gdp_per_capita']], on=['country','year'])
merged = merged.dropna()

plt.figure(figsize=(9,5))
plt.scatter(merged['gdp_per_capita'], merged['adult_literacy_rate'], alpha=0.4, color='teal', s=15)
plt.xscale('log')
plt.xlabel('GDP per Capita (log scale)')
plt.ylabel('Adult Literacy Rate (%)')
plt.title('GDP per Capita vs Adult Literacy Rate')
plt.tight_layout()
plt.show()
print('📌 Insight: Higher GDP tends to correlate with higher literacy rates.')

In [ ]:
# ── 5D. Gender Gap in Youth Literacy ──
if 'literacy_gender_gap' in df_literacy.columns:
    gap_2020 = df_literacy[df_literacy['year']==2020][['country','literacy_gender_gap']].dropna()
    gap_2020 = gap_2020.sort_values('literacy_gender_gap', ascending=False).head(15)
    plt.figure(figsize=(10,5))
    plt.barh(gap_2020['country'], gap_2020['literacy_gender_gap'], color='salmon')
    plt.xlabel('Gender Gap (Male - Female Youth Literacy %)')
    plt.title('Top 15 Countries by Youth Literacy Gender Gap (2020)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5E. Global Literacy Trend Over Time ──
global_trend = df_literacy.groupby('year')['adult_literacy_rate'].mean().reset_index()
plt.figure(figsize=(10,4))
plt.plot(global_trend['year'], global_trend['adult_literacy_rate'], marker='o', color='royalblue', linewidth=2)
plt.title('Global Average Adult Literacy Rate Over Time')
plt.xlabel('Year')
plt.ylabel('Avg Literacy Rate (%)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5F. Correlation Heatmap ──
merged_all = pd.merge(df_literacy[['country','year','adult_literacy_rate','literacy_gender_gap','youth_literacy_avg']],
                      df_gdp_schooling[['country','year','gdp_per_capita','avg_years_schooling']], on=['country','year'], how='inner')
corr_cols = ['adult_literacy_rate','literacy_gender_gap','youth_literacy_avg','gdp_per_capita','avg_years_schooling']
corr_cols = [c for c in corr_cols if c in merged_all.columns]
corr = merged_all[corr_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap — Literacy, GDP & Schooling')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5G. Top 10 & Bottom 10 Countries by Adult Literacy (Latest Year) ──
latest_year = df_literacy['year'].max()
latest_lit  = df_literacy[df_literacy['year']==latest_year][['country','adult_literacy_rate']].dropna()
top10    = latest_lit.nlargest(10,  'adult_literacy_rate')
bottom10 = latest_lit.nsmallest(10, 'adult_literacy_rate')

fig, axes = plt.subplots(1,2, figsize=(14,5))
axes[0].barh(top10['country'],    top10['adult_literacy_rate'],    color='mediumseagreen')
axes[0].set_title(f'Top 10 Literacy Countries ({latest_year})')
axes[0].invert_yaxis()
axes[1].barh(bottom10['country'], bottom10['adult_literacy_rate'], color='tomato')
axes[1].set_title(f'Bottom 10 Literacy Countries ({latest_year})')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

---
## 🗄️ Step 6: SQL Database Creation & Data Insertion

In [ ]:
conn   = sqlite3.connect('literacy_data.db')
cursor = conn.cursor()

cursor.executescript('''
    DROP TABLE IF EXISTS literacy_rates;
    DROP TABLE IF EXISTS illiteracy_population;
    DROP TABLE IF EXISTS gdp_schooling;
''')

# Table 1: literacy_rates
cursor.execute('''
    CREATE TABLE literacy_rates (
        country               VARCHAR(100),
        code                  VARCHAR(10),
        year                  INT,
        adult_literacy_rate   DECIMAL(6,2),
        youth_literacy_male   DECIMAL(6,2),
        youth_literacy_female DECIMAL(6,2),
        literacy_gender_gap   DECIMAL(6,2),
        youth_literacy_avg    DECIMAL(6,2),
        literacy_growth_rate  DECIMAL(8,4),
        PRIMARY KEY (country, year)
    )
''')

# Table 2: illiteracy_population
cursor.execute('''
    CREATE TABLE illiteracy_population (
        country           VARCHAR(100),
        code              VARCHAR(10),
        year              INT,
        illiterate_total  BIGINT,
        illiterate_male   BIGINT,
        illiterate_female BIGINT,
        literate_total    BIGINT,
        total_population  BIGINT,
        illiteracy_pct    DECIMAL(6,2),
        PRIMARY KEY (country, year)
    )
''')

# Table 3: gdp_schooling
cursor.execute('''
    CREATE TABLE gdp_schooling (
        country                VARCHAR(100),
        code                   VARCHAR(10),
        year                   INT,
        gdp_per_capita         DECIMAL(12,2),
        avg_years_schooling    DECIMAL(6,2),
        gdp_per_schooling_year DECIMAL(12,2),
        PRIMARY KEY (country, year)
    )
''')

conn.commit()
print('✅ All 3 tables created!')

In [ ]:
# Insert data — only insert columns that exist in both df and table
def safe_insert(df, table, conn):
    cursor = conn.cursor()
    cursor.execute(f'PRAGMA table_info({table})')
    table_cols = [row[1] for row in cursor.fetchall()]
    common_cols = [c for c in df.columns if c in table_cols]
    df[common_cols].to_sql(table, conn, if_exists='append', index=False)
    print(f'✅ {table}: {len(df)} rows inserted')

safe_insert(df_literacy,    'literacy_rates',       conn)
safe_insert(df_illiteracy,  'illiteracy_population', conn)
safe_insert(df_gdp_schooling,'gdp_schooling',        conn)

conn.commit()
print('\n🎉 All data inserted into literacy_data.db!')

---
## 🔎 Step 7: SQL Queries (13 Total)

In [ ]:
def run_query(title, sql):
    print(f'\n{"="*60}')
    print(f'📌 {title}')
    print('='*60)
    df = pd.read_sql_query(sql, conn)
    print(df.to_string(index=False))
    return df

In [ ]:
# Q1: Top 5 countries with highest adult literacy in 2020
run_query('Q1: Top 5 Countries — Highest Adult Literacy (2020)', '''
    SELECT country, ROUND(adult_literacy_rate,2) AS adult_literacy_rate
    FROM literacy_rates
    WHERE year = 2020 AND adult_literacy_rate IS NOT NULL
    ORDER BY adult_literacy_rate DESC
    LIMIT 5
''')

In [ ]:
# Q2: Countries where female youth literacy < 80%
run_query('Q2: Countries Where Female Youth Literacy < 80%', '''
    SELECT country, year, ROUND(youth_literacy_female,2) AS youth_literacy_female
    FROM literacy_rates
    WHERE youth_literacy_female < 80 AND youth_literacy_female IS NOT NULL
    ORDER BY youth_literacy_female ASC
    LIMIT 15
''')

In [ ]:
# Q3: Average adult literacy per continent (using code as proxy)
run_query('Q3: Average Adult Literacy by Region (Latest Year Per Country)', '''
    SELECT SUBSTR(code,1,1) AS region_prefix,
           COUNT(DISTINCT country) AS countries,
           ROUND(AVG(adult_literacy_rate),2) AS avg_literacy
    FROM literacy_rates
    WHERE adult_literacy_rate IS NOT NULL
    GROUP BY region_prefix
    ORDER BY avg_literacy DESC
''')

In [ ]:
# Q4: Countries with illiteracy % > 20% in 2000
run_query('Q4: Countries With Illiteracy > 20% in 2000', '''
    SELECT country, year, ROUND(illiteracy_pct,2) AS illiteracy_pct
    FROM illiteracy_population
    WHERE year = 2000 AND illiteracy_pct > 20
    ORDER BY illiteracy_pct DESC
    LIMIT 15
''')

In [ ]:
# Q5: Trend of illiteracy % for India 2000-2020
run_query('Q5: Illiteracy % Trend for India (2000-2020)', '''
    SELECT year, ROUND(illiteracy_pct,2) AS illiteracy_pct
    FROM illiteracy_population
    WHERE country = 'India' AND year BETWEEN 2000 AND 2020
    ORDER BY year
''')

In [ ]:
# Q6: Top 10 countries with largest illiterate population (latest year)
run_query('Q6: Top 10 Countries by Illiterate Population (Latest Year)', '''
    SELECT country, year, illiterate_total
    FROM illiteracy_population
    WHERE year = (SELECT MAX(year) FROM illiteracy_population)
      AND illiterate_total IS NOT NULL
    ORDER BY illiterate_total DESC
    LIMIT 10
''')

In [ ]:
# Q7: Countries with avg_years_schooling > 7 and gdp_per_capita < 5000
run_query('Q7: High Schooling but Low GDP Countries', '''
    SELECT country, year,
           ROUND(avg_years_schooling,2) AS avg_years_schooling,
           ROUND(gdp_per_capita,2) AS gdp_per_capita
    FROM gdp_schooling
    WHERE avg_years_schooling > 7 AND gdp_per_capita < 5000
    ORDER BY avg_years_schooling DESC
    LIMIT 15
''')

In [ ]:
# Q8: Rank countries by GDP per schooling year (2020)
run_query('Q8: Countries Ranked by GDP per Schooling Year (2020)', '''
    SELECT country,
           ROUND(gdp_per_capita,2) AS gdp_per_capita,
           ROUND(avg_years_schooling,2) AS avg_years_schooling,
           ROUND(gdp_per_schooling_year,2) AS gdp_per_schooling_year,
           RANK() OVER (ORDER BY gdp_per_schooling_year DESC) AS rank
    FROM gdp_schooling
    WHERE year = 2020 AND gdp_per_schooling_year IS NOT NULL
    LIMIT 15
''')

In [ ]:
# Q9: Global average schooling years per year
run_query('Q9: Global Average Schooling Years Per Year', '''
    SELECT year, ROUND(AVG(avg_years_schooling),2) AS global_avg_schooling
    FROM gdp_schooling
    WHERE avg_years_schooling IS NOT NULL
    GROUP BY year
    ORDER BY year
''')

In [ ]:
# Q10: JOIN — Top 10 countries with highest GDP but low schooling (< 6 years) in 2020
run_query('Q10: High GDP but Low Schooling Countries (2020)', '''
    SELECT g.country,
           ROUND(g.gdp_per_capita,2) AS gdp_per_capita,
           ROUND(g.avg_years_schooling,2) AS avg_years_schooling
    FROM gdp_schooling g
    WHERE g.year = 2020
      AND g.avg_years_schooling < 6
      AND g.gdp_per_capita IS NOT NULL
    ORDER BY g.gdp_per_capita DESC
    LIMIT 10
''')

In [ ]:
# Q11: JOIN — High illiterate population despite >10 avg schooling years
run_query('Q11: High Illiteracy Despite 10+ Years Schooling', '''
    SELECT i.country, i.year,
           i.illiterate_total,
           ROUND(g.avg_years_schooling,2) AS avg_years_schooling
    FROM illiteracy_population i
    JOIN gdp_schooling g ON i.country = g.country AND i.year = g.year
    WHERE g.avg_years_schooling > 10
      AND i.illiterate_total > 1000000
    ORDER BY i.illiterate_total DESC
    LIMIT 10
''')

In [ ]:
# Q12: JOIN — Literacy & GDP trend for India over last 20 years
run_query('Q12: Literacy vs GDP Growth — India (Last 20 Years)', '''
    SELECT l.country, l.year,
           ROUND(l.adult_literacy_rate,2) AS adult_literacy_rate,
           ROUND(g.gdp_per_capita,2) AS gdp_per_capita
    FROM literacy_rates l
    JOIN gdp_schooling g ON l.country = g.country AND l.year = g.year
    WHERE l.country = 'India' AND l.year >= 2000
    ORDER BY l.year
''')

In [ ]:
# Q13: JOIN — Youth literacy gender gap for countries with GDP > $30,000 in 2020
run_query('Q13: Youth Literacy Gender Gap — High GDP Countries (2020)', '''
    SELECT l.country,
           ROUND(l.youth_literacy_male,2)   AS youth_male,
           ROUND(l.youth_literacy_female,2) AS youth_female,
           ROUND(l.literacy_gender_gap,2)   AS gender_gap,
           ROUND(g.gdp_per_capita,2)        AS gdp_per_capita
    FROM literacy_rates l
    JOIN gdp_schooling g ON l.country = g.country AND l.year = g.year
    WHERE g.year = 2020
      AND g.gdp_per_capita > 30000
      AND l.youth_literacy_male IS NOT NULL
      AND l.youth_literacy_female IS NOT NULL
    ORDER BY ABS(l.literacy_gender_gap) DESC
    LIMIT 15
''')

---
## ✅ Step 8: Verify & Download Database

In [ ]:
print('📊 Final Row Counts:')
for t in ['literacy_rates','illiteracy_population','gdp_schooling']:
    count = cursor.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'   {t:<30} → {count:,} rows')
conn.close()
print('\n✅ Database connection closed.')

In [ ]:
from google.colab import files
files.download('literacy_data.db')
print('✅ literacy_data.db downloaded! Use it in your Streamlit app.')